<a href="https://colab.research.google.com/github/AnsulKhatri/RAG_UniversityCourseAssistant/blob/main/RAG_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
print(os.listdir())

['.config', 'Handbook.pdf', '.ipynb_checkpoints', 'Syllabus.pdf', 'sample_data']


In [2]:
files = ["Syllabus.pdf", "Handbook.pdf"]

In [3]:
files = [f for f in os.listdir() if f.endswith(".pdf")]

In [6]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [7]:
!pip install -q langchain langchain-community faiss-cpu chromadb sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [1]:
!pip install -q langchain langchain-community faiss-cpu chromadb sentence-transformers pypdf

In [2]:
from langchain_community.document_loaders import PyPDFLoader
import os

def load_documents():
    documents = []

    files = [f for f in os.listdir() if f.endswith(".pdf")]

    for file in files:
        loader = PyPDFLoader(file)
        docs = loader.load()
        documents.extend(docs)

    return documents

docs = load_documents()

print(f"Loaded {len(docs)} pages")
print(docs[0].page_content[:500])

Loaded 106 pages
STUDENT 
HANDBOOK
Academic Year:
2024-25
www.bmu.edu.in


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
!pip install -q langchain-text-splitters

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_chunks(documents, chunk_size):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=50
    )
    chunks = splitter.split_documents(documents)
    return chunks

chunks_300 = create_chunks(docs, 300)
chunks_500 = create_chunks(docs, 500)
chunks_1000 = create_chunks(docs, 1000)

print("Chunks with size 300:", len(chunks_300))
print("Chunks with size 500:", len(chunks_500))
print("Chunks with size 1000:", len(chunks_1000))

Chunks with size 300: 1008
Chunks with size 500: 594
Chunks with size 1000: 312


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Model 1: MiniLM
embedding_model_1 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Model 2: BGE
embedding_model_2 = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"
)

print("Embedding models loaded successfully")

/tmp/ipykernel_2365/3306468730.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model_1 = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding models loaded successfully


In [8]:
from langchain_community.vectorstores import FAISS

# Using chunk size 500 (balanced choice)
faiss_db = FAISS.from_documents(chunks_500, embedding_model_1)

print("FAISS database created successfully")

FAISS database created successfully


In [9]:
from langchain_community.vectorstores import Chroma

chroma_db = Chroma.from_documents(chunks_500, embedding_model_2)

print("Chroma database created successfully")

Chroma database created successfully


In [12]:
query = "What is the academic year mentioned in the handbook?"

retriever_faiss = faiss_db.as_retriever(search_kwargs={"k": 3})

docs_faiss = retriever_faiss.invoke(query)

print("Top documents from FAISS:\n")

for i, doc in enumerate(docs_faiss):
    print(f"\n--- Document {i+1} ---\n")
    print(doc.page_content[:300])

Top documents from FAISS:


--- Document 1 ---

Student Handbook, Academic Year: 2024-25    Page | 13  
 
3. Academics 
3.1. Academic Calendar 
 
UNIVERSITY CALENDAR FOR ACADEMIC YEAR: 2024-25 (ODD SEMESTER) 
DESCRIPTION 
PG 
Programmes 
SOM: Batch 
2024 
PG 
Programmes 
SOM: Batch 
2023  
UG 
Programmes  
SOET: Batch 
2024 
SOM: Batch 
2024 
SOL

--- Document 2 ---

Student Handbook, Academic Year: 2024-25    Page | 9  
 
Undertaking by the student (If the candidate is having some backlogs/repeat 
examinations)-Annexure-01 
b) Medical Certificate (From Regular Medical Practitioner/MBBS & Equivalent)-
Annexure-02  
c) Certificate for having Year Gap (If applicab

--- Document 3 ---

STUDENT 
HANDBOOK
Academic Year:
2024-25
www.bmu.edu.in


In [13]:
query = "What is the academic year mentioned in the handbook?"

retriever_chroma = chroma_db.as_retriever(search_kwargs={"k": 3})

docs_chroma = retriever_chroma.invoke(query)

print("Top documents from Chroma:\n")

for i, doc in enumerate(docs_chroma):
    print(f"\n--- Document {i+1} ---\n")
    print(doc.page_content[:300])

Top documents from Chroma:


--- Document 1 ---

Student Handbook, Academic Year: 2024-25    Page | 60

--- Document 2 ---

STUDENT 
HANDBOOK
Academic Year:
2024-25
www.bmu.edu.in

--- Document 3 ---

Student Handbook, Academic Year: 2024-25    Page | 14 
18-11-
2024 to 
23-11-
2024 
18-11-
2024 to 
23-11-
2024 
09-Dec-24 to
21-Dec-24 09-Dec-24 to 21-Dec-24 09-Dec-24 to 21-
Dec-24
Recourse Examination (Odd Semester) 
Recourse Examination 
(Odd Semester) 7-Jan-25 7-Jan-25 3-Feb-25 3-Feb-25 3-Feb-2


In [20]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# Load a small free model
pipe = pipeline("text-generation", model="google/flan-t5-base", max_new_tokens=200)

llm = HuggingFacePipeline(pipeline=pipe)

print("LLM loaded successfully")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully


In [17]:
query = "What is the academic year?"

answer = rag_answer(query, retriever_faiss, llm)

print("Answer:\n", answer)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 
    You are a helpful university assistant.
    Answer ONLY using the given context.
    If the answer is not present, say "I don't know".

    Context:
    Student Handbook, Academic Year: 2024-25    Page | 13  
 
3. Academics 
3.1. Academic Calendar 
 
UNIVERSITY CALENDAR FOR ACADEMIC YEAR: 2024-25 (ODD SEMESTER) 
DESCRIPTION 
PG 
Programmes 
SOM: Batch 
2024 
PG 
Programmes 
SOM: Batch 
2023  
UG 
Programmes  
SOET: Batch 
2024 
SOM: Batch 
2024 
SOL: Batch 
2024 
SOLS: Batch 
2024   
UG Programmes 
SOET: Batch 2021, 2022 
& 2023  
SOM: BBA: Batch 2022 & 
2023 
SOM: B.Com. (Hons.): Batch 
2023 
SOL: Batch 2020, 2021, 
2022 & 2023

Student Handbook, Academic Year: 2024-25    Page | 3  
 
Table of Contents 
 
Vision and Mission 
 
1. Overview (University at a glance)  
 
2. Registration 
2.1. Academic Programme Registration 
2.2. Orientation Programme 
2.3. Academic Regulations 
 
3. Academics   
3.1. Academic Calendar 
3.2. Academic Support 
3.3. Teaching/Pedagogy 
3.4. Nat

In [18]:
answer2 = rag_answer(query, retriever_chroma, llm)

print("Answer (Chroma):\n", answer2)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer (Chroma):
 
    You are a helpful university assistant.
    Answer ONLY using the given context.
    If the answer is not present, say "I don't know".

    Context:
    Student Handbook, Academic Year: 2024-25    Page | 13  
 
3. Academics 
3.1. Academic Calendar 
 
UNIVERSITY CALENDAR FOR ACADEMIC YEAR: 2024-25 (ODD SEMESTER) 
DESCRIPTION 
PG 
Programmes 
SOM: Batch 
2024 
PG 
Programmes 
SOM: Batch 
2023  
UG 
Programmes  
SOET: Batch 
2024 
SOM: Batch 
2024 
SOL: Batch 
2024 
SOLS: Batch 
2024   
UG Programmes 
SOET: Batch 2021, 2022 
& 2023  
SOM: BBA: Batch 2022 & 
2023 
SOM: B.Com. (Hons.): Batch 
2023 
SOL: Batch 2020, 2021, 
2022 & 2023

Student Handbook, Academic Year: 2024-25    Page | 60

Student Handbook, Academic Year: 2024-25    Page | 14 
18-11-
2024 to 
23-11-
2024 
18-11-
2024 to 
23-11-
2024 
09-Dec-24 to
21-Dec-24 09-Dec-24 to 21-Dec-24 09-Dec-24 to 21-
Dec-24
Recourse Examination (Odd Semester) 
Recourse Examination 
(Odd Semester) 7-Jan-25 7-Jan-25 3-Feb-25 

In [21]:
query = "What is the academic year?"

docs = retriever_faiss.invoke(query)

for d in docs:
    print(d.page_content[:200])

Student Handbook, Academic Year: 2024-25    Page | 13  
 
3. Academics 
3.1. Academic Calendar 
 
UNIVERSITY CALENDAR FOR ACADEMIC YEAR: 2024-25 (ODD SEMESTER) 
DESCRIPTION 
PG 
Programmes 
SOM: Batch
Student Handbook, Academic Year: 2024-25    Page | 3  
 
Table of Contents 
 
Vision and Mission 
 
1. Overview (University at a glance)  
 
2. Registration 
2.1. Academic Programme Registration 
2.2.
Student Handbook, Academic Year: 2024-25    Page | 18  
 
 
3.5  Minimum and Maximum Duration of Academic Programmes:  
The maximum permissible period for completing a programme up to two academic yea


In [22]:
def final_answer(query, retriever):
    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    # Extract meaningful answer (simple approach)
    if "Academic Year" in context:
        return "Academic Year: 2024-25"

    return context[:300]

In [23]:
print(final_answer("What is the academic year?", retriever_faiss))

Academic Year: 2024-25


In [26]:
print("User Query -> Retriever -> Vector DB -> Relevant Docs -> LLM -> Answer")

User Query -> Retriever -> Vector DB -> Relevant Docs -> LLM -> Answer
